In [1]:
import pandas as pd
from pathlib import Path

In [14]:
FILE_PATH = Path('../../data/customer_churn_records.csv')
df = pd.read_csv(FILE_PATH)
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425


# Group By
O `GROUP BY` funciona da seguinte forma:
1. Agrupa os iguais;
2. Passa uma função resumo, por exemplo:
    - `contagem`, `soma`, `media`, `mediana`, `std`, etc. Nos grupos criados

#### 1) Qual a média de credit_score e balance por geography?

In [15]:
df.groupby('Geography')[['CreditScore', 'Balance']].mean()

,CreditScore,Balance
Geography,,
France,649.668329,62092.636516
Germany,651.453567,119730.116134
Spain,651.333872,61818.147763


#### 2) Qual a taxa de saída (exited) por gender e geography?

In [17]:
df.groupby(['Gender', 'Geography'])['Exited'].mean()

Gender  Geography
Female  France       0.203450
        Germany      0.375524
        Spain        0.212121
Male    France       0.127497
        Germany      0.278116
        Spain        0.131124
Name: Exited, dtype: float64

#### 3) Clientes com mais produtos (num_of_products) têm maior saldo médio e maior taxa de churn?

In [18]:
df.groupby('NumOfProducts').agg({
    'Balance': 'mean',
    'Exited': 'mean'
})

,Balance,Exited
NumOfProducts,,
1,98551.870614,0.277144
2,51879.145813,0.076035
3,75458.328195,0.827068
4,93733.135000,1.000000


#### 4) Qual a diferença de salário médio e score de crédito entre clientes ativos e inativos?

In [19]:
df.groupby('IsActiveMember').agg({
    'EstimatedSalary': 'mean',
    'CreditScore': 'mean'
})

,EstimatedSalary,CreditScore
IsActiveMember,,
0,100767.203854,647.973603
1,99452.965894,652.934188


#### 5) Para cada card_type, qual o perfil médio dos clientes que reclamaram (complain = 1)?

In [26]:
(
    df[df['Complain'] == 1]
    .groupby('Card Type').agg({
        'CreditScore': 'mean',
        'Balance': 'mean',
        'Satisfaction Score': 'mean',
        'Card Type': 'count'
    })
    .sort_index()
)

,CreditScore,Balance,Satisfaction Score,Card Type
Card Type,,,,
DIAMOND,643.001828,92092.780091,3.014625,547
GOLD,647.297521,89519.103967,3.076446,484
PLATINUM,650.307241,90265.597808,2.972603,511
SILVER,641.071713,92594.866295,2.940239,502


#### 6) Após agrupar passe uma função personalizada nos grupos

In [2]:
df = pd.DataFrame({
    "regiao": ["norte", "norte", "sul", "sul", "sul"],
    "produto": ["A", "A", "A", "B", "B"],
    "y": [1500, 1200, 800, 200, 2200],
    "y_hat": [110, 1100, 990, 100, 2500],
})
print(df)

  regiao produto     y  y_hat
0  norte       A  1500    110
1  norte       A  1200   1100
2    sul       A   800    990
3    sul       B   200    100
4    sul       B  2200   2500


In [13]:
def func(grupo: pd.DataFrame) -> pd.Series:
    erro = grupo["y"] - grupo["y_hat"]
    mae = erro.abs().mean()
    rmse = (erro ** 2).mean() ** (1 / 2)
    mape = (erro.abs() / grupo["y"].abs()).mean().round(2)
    qtde = len(grupo)
    
    return pd.Series({
        "mae": mae,
        "rmse": rmse,
        "mape": mape,
        "qtde": qtde,
    })

In [14]:
(
    df
    .groupby(["regiao", "produto"])
    .apply(func, include_groups=False)
    .reset_index()
)

,regiao,produto,mae,rmse,mape,qtde
0,norte,A,745.0,985.418693,0.50,2.0
1,sul,A,190.0,190.000000,0.24,1.0
2,sul,B,200.0,223.606798,0.32,2.0


In [15]:
(
    df
    .groupby(["regiao", "produto"])
    .agg(
        amplitude=("y", lambda s: s.max() - s.min()),
        agg_custom=("y_hat", lambda s: (s / 1000).mean()),
        qtde=("y", "size"),
    )
    .reset_index()
)

,regiao,produto,amplitude,agg_custom,qtde
0,norte,A,300,0.605,2
1,sul,A,0,0.990,1
2,sul,B,2000,1.300,2
